# Muse EEG Heads — Night holdout (SC4001 → SC4002)

Frozen **CBraMod** + tiny **Head A** binary (`drowsy` vs `hypnagogic`).

**Protocol:** train on SC4001 N1-slice (balanced undersample); evaluate on **different night** SC4002 (all labeled + balanced subsample). Same-night SC4001 val for comparison.

**Slice recipe:** `around_stage='stage 1'`, pre=20 min, post=40 min; window 2 s, hop 0.5 s; majority 0.7. Labels: W→drowsy, N1→hypnagogic.

**Honest caveat:** Sleep-EDF 2-ch Muse-proxy; domain gap vs TUEG pretrain. Macro-F1 on imbalanced all-labeled holdout can look different from balanced accuracy.

**Licenses:** Sleep-EDF PhysioNet **ODC-By**; CBraMod **Apache-2.0**. NO LUNA / L-FAME / SEED-VIG.


## 1. Setup


In [ ]:
# Setup — helpers from muse-eeg-heads-src; prefer uv
import os, sys, json, hashlib
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone

WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".").resolve()
INPUT_ROOT = Path("/kaggle/input")
ROOT_CANDS = [
    Path("/kaggle/working"),
    Path("/workspace/muse-eeg-heads"),
    Path(".").resolve(),
]

def find_src_dir():
    if INPUT_ROOT.exists():
        for p in INPUT_ROOT.rglob("sleep_edf.py"):
            return p.parent
        for name in ("muse-eeg-heads-src",):
            direct = INPUT_ROOT / name
            if direct.exists():
                return direct
    for r in ROOT_CANDS:
        src = r / "src"
        if (src / "sleep_edf.py").exists():
            return src
    raise FileNotFoundError("src not found")

SRC = find_src_dir()
# package layout (.../muse-eeg-heads/src) vs flat dataset (.../muse-eeg-heads-src/*.py)
if SRC.name == "src":
    sys.path.insert(0, str(SRC.parent))
else:
    sys.path.insert(0, str(SRC))

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

try:
    from src.sleep_edf import (
        PROXY_NOTE, STAGE_TO_HEAD_A, find_pilot_pairs,
        load_sleep_edf_recording, windows_with_head_a_labels,
    )
    from src.cbramod_encoder import FrozenCBraModEncoder
    from src.head_a import (
        HEAD_A_BINARY_LABELS, HeadALinear, class_weights_from_y,
        labels_to_ids, undersample_balanced,
    )
    from src.metrics import confusion_matrix, macro_f1, per_class_report
except ImportError:
    from sleep_edf import (
        PROXY_NOTE, STAGE_TO_HEAD_A, find_pilot_pairs,
        load_sleep_edf_recording, windows_with_head_a_labels,
    )
    from cbramod_encoder import FrozenCBraModEncoder
    from head_a import (
        HEAD_A_BINARY_LABELS, HeadALinear, class_weights_from_y,
        labels_to_ids, undersample_balanced,
    )
    from metrics import confusion_matrix, macro_f1, per_class_report

SEED = 42
WINDOW_SEC, HOP_SEC, TARGET_SR = 2.0, 0.5, 256.0
PRE_SEC, POST_SEC, MAJORITY = 20 * 60, 40 * 60, 0.7
EPOCHS, BATCH, LR, VAL_FRAC = 5, 32, 1e-3, 0.2
print("SRC", SRC)
print("labels", HEAD_A_BINARY_LABELS)


## 2. Locate data + weights (or prebuilt windows)


In [ ]:
def find_pilot_root():
    cands = []
    if INPUT_ROOT.exists():
        for p in INPUT_ROOT.rglob("sleep-edfx-pilot"):
            cands.append(p)
        for p in INPUT_ROOT.rglob("*-PSG.edf"):
            cands.append(p.parent.parent if p.parent.name == "sleep-cassette" else p.parent)
    cands += [
        Path("/tmp/kaggle_out3/data/sleep-edfx-pilot"),
        Path("/workspace/muse-eeg-heads/kaggle_datasets/muse-eeg-heads-cache/data/sleep-edfx-pilot"),
    ]
    for r in cands:
        if r and Path(r).exists() and find_pilot_pairs(r):
            return Path(r)
    return None

def find_weights():
    cands = []
    if INPUT_ROOT.exists():
        for p in INPUT_ROOT.rglob("pretrained_weights.pth"):
            cands.append(p)
    cands += [
        Path("/tmp/kaggle_out3/models/CBraMod/pretrained_weights.pth"),
        Path("/workspace/muse-eeg-heads/kaggle_datasets/muse-eeg-heads-cache/models/CBraMod/pretrained_weights.pth"),
    ]
    for p in cands:
        if p.exists():
            return p
    raise FileNotFoundError("CBraMod weights missing")

def find_prebuilt(tag):
    name = f"sleep_edf_{tag}_n1slice_windows.npz"
    cands = []
    if INPUT_ROOT.exists():
        for p in INPUT_ROOT.rglob(name):
            cands.append(p)
    cands += [
        Path(f"/workspace/muse-eeg-heads/exports/windows_{tag}") / name,
        Path("/workspace/muse-eeg-heads/kaggle_datasets/muse-eeg-heads-windows") / name,
        WORKING / name,
    ]
    for p in cands:
        if p.exists():
            return p
    return None

pilot = find_pilot_root()
weights = find_weights()
print("pilot", pilot)
print("weights", weights)
print("prebuilt sc4001", find_prebuilt("sc4001"))
print("prebuilt sc4002", find_prebuilt("sc4002"))


## 3. Build or load windows (SC4001 + SC4002)


In [ ]:
def pair_for(pairs, subject):
    for psg, hyp in pairs:
        if subject in psg.name:
            return psg, hyp
    raise FileNotFoundError(subject)

def build_or_load(tag, subject, out_dir: Path):
    pre = find_prebuilt(tag)
    if pre is not None:
        z = np.load(pre, allow_pickle=True)
        X, y = z["X"], z["y"]
        names = [str(x) for x in z["label_names"].tolist()]
        assert names == HEAD_A_BINARY_LABELS
        print(f"loaded {pre} shape={X.shape} counts={dict(Counter(y.tolist()))}")
        meta = {"psg_file": f"{subject}*", "slice_start_sec": None, "source": str(pre)}
        man = pre.with_name(pre.name.replace("_windows.npz", "_manifest.json"))
        if not man.exists():
            # package may use sleep_edf_sc4001_n1slice_manifest.json
            alt = pre.parent / f"sleep_edf_{tag}_n1slice_manifest.json"
            man = alt if alt.exists() else man
        if man.exists():
            meta = json.loads(man.read_text())
            meta["source"] = str(pre)
        return X, y, meta

    assert pilot is not None, "Need pilot Sleep-EDF or prebuilt windows"
    pairs = find_pilot_pairs(pilot)
    psg, hyp = pair_for(pairs, subject)
    rec = load_sleep_edf_recording(
        psg, hyp, target_sr=TARGET_SR,
        around_stage="stage 1", pre_sec=PRE_SEC, post_sec=POST_SEC,
    )
    X, labels, keep = windows_with_head_a_labels(
        rec["data"], rec["stages"], sfreq=TARGET_SR,
        window_sec=WINDOW_SEC, hop_sec=HOP_SEC, majority_frac=MAJORITY,
    )
    mask = [lab in HEAD_A_BINARY_LABELS for lab in labels]
    X = X[np.asarray(mask)]
    labels = [lab for lab, m in zip(labels, mask) if m]
    y = labels_to_ids(labels, HEAD_A_BINARY_LABELS)
    hop = int(round(HOP_SEC * TARGET_SR))
    starts = np.asarray([keep[i] * hop for i, m in enumerate(mask) if m], dtype=np.int64)
    out_dir.mkdir(parents=True, exist_ok=True)
    npz = out_dir / f"sleep_edf_{tag}_n1slice_windows.npz"
    np.savez_compressed(npz, X=X.astype(np.float32), y=y, starts=starts,
                        label_names=np.asarray(HEAD_A_BINARY_LABELS))
    meta = {
        "psg_file": psg.name, "hypno_file": hyp.name,
        "slice_start_sec": rec["slice_start_sec"],
        "n_windows_per_label": dict(Counter(labels)),
        "n_windows_total": int(X.shape[0]),
        "npz_path": str(npz),
        "source": "built",
    }
    (out_dir / f"sleep_edf_{tag}_n1slice_manifest.json").write_text(json.dumps(meta, indent=2))
    print(f"built {npz} shape={X.shape} counts={meta['n_windows_per_label']} slice_start={meta['slice_start_sec']}")
    return X, y, meta

out1 = WORKING / "windows_sc4001"
out2 = WORKING / "windows_sc4002"
X1, y1, meta1 = build_or_load("sc4001", "SC4001", out1)
X2, y2, meta2 = build_or_load("sc4002", "SC4002", out2)
print("SC4001 meta slice_start_sec", meta1.get("slice_start_sec"), "counts", meta1.get("n_windows_per_label"))
print("SC4002 meta slice_start_sec", meta2.get("slice_start_sec"), "counts", meta2.get("n_windows_per_label"))
if int(Counter(y2.tolist()).get(1, 0)) < 50:
    print("NOTE: few hypnagogic windows on SC4002 — interpret metrics with care; labels not invented.")


## 4. Freeze CBraMod; train Head A on balanced SC4001


In [ ]:
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
Xb, yb = undersample_balanced(X1, y1, rng)
n_bal = len(yb)
perm = rng.permutation(n_bal)
n_val = max(1, int(round(n_bal * VAL_FRAC)))
val_idx, tr_idx = perm[:n_val], perm[n_val:]
X_tr, y_tr = Xb[tr_idx], yb[tr_idx]
X_val, y_val = Xb[val_idx], yb[val_idx]
print("balanced", Xb.shape, "train", X_tr.shape, "val", X_val.shape)

device = torch.device("cpu")
encoder = FrozenCBraModEncoder(weights, source_sr=TARGET_SR, pool="mean").to(device)
print(encoder.adapter_notes())

def encode_all(X):
    if len(X) == 0:
        return torch.zeros((0, 200))
    Xt = torch.from_numpy(X.astype(np.float32))
    outs = []
    with torch.no_grad():
        for i in range(0, len(Xt), BATCH):
            outs.append(encoder(Xt[i:i+BATCH].to(device)).cpu())
    return torch.cat(outs, 0)

emb_tr = encode_all(X_tr)
emb_val = encode_all(X_val)
emb2 = encode_all(X2)
counts2 = Counter(y2.tolist())
n_min2 = min(counts2.get(0, 0), counts2.get(1, 0))
if n_min2 == 0:
    X2b = np.zeros((0, *X2.shape[1:]), np.float32); y2b = np.zeros((0,), np.int64)
    emb2b = torch.zeros((0, 200))
    print("LIMITATION: cannot balance SC4002", dict(counts2))
else:
    X2b, y2b = undersample_balanced(X2, y2, rng)
    emb2b = encode_all(X2b)
    print("SC4002 balanced", X2b.shape, Counter(y2b.tolist()))

head = HeadALinear(in_dim=emb_tr.shape[-1], n_classes=2).to(device)
crit = nn.CrossEntropyLoss(weight=class_weights_from_y(y_tr, 2).to(device))
opt = torch.optim.Adam(head.parameters(), lr=LR)
loader = DataLoader(TensorDataset(emb_tr, torch.from_numpy(y_tr)), batch_size=BATCH, shuffle=True)
history = []
for ep in range(EPOCHS):
    head.train(); total = n = 0
    for xb, yb_ in loader:
        opt.zero_grad()
        loss = crit(head(xb.to(device)), yb_.to(device))
        loss.backward(); opt.step()
        total += float(loss.item()) * len(yb_); n += len(yb_)
    head.eval()
    with torch.no_grad():
        pred = head(emb_tr.to(device)).argmax(-1).cpu().numpy()
    row = {"epoch": ep+1, "loss": total/max(n,1),
           "train_acc": float((pred == y_tr).mean()),
           "train_macro_f1": macro_f1(y_tr.tolist(), pred.tolist(), HEAD_A_BINARY_LABELS)}
    history.append(row); print(row)


## 5. Evaluate — same-night val + SC4002 holdout


In [ ]:
def eval_split(emb, y, name):
    head.eval()
    if len(y) == 0:
        print(name, "EMPTY"); return {"name": name, "n": 0, "acc": None, "macro_f1": None}
    with torch.no_grad():
        pred = head(emb.to(device)).argmax(-1).cpu().numpy()
    acc = float((pred == y).mean())
    f1 = macro_f1(y.tolist(), pred.tolist(), HEAD_A_BINARY_LABELS)
    rep = per_class_report(y.tolist(), pred.tolist(), HEAD_A_BINARY_LABELS)
    cm = confusion_matrix(y.tolist(), pred.tolist(), HEAD_A_BINARY_LABELS)
    per = {k: v for k, v in rep.items() if k != "macro_f1"}
    out = {"name": name, "n": int(len(y)), "acc": acc, "macro_f1": f1,
           "per_class": per, "confusion": cm.tolist(),
           "counts_true": {HEAD_A_BINARY_LABELS[i]: int((y == i).sum()) for i in range(2)}}
    print(f"\n=== {name} === n={out['n']} acc={acc:.4f} macro_f1={f1:.4f}")
    print("per_class", json.dumps(per, indent=2))
    print("confusion rows=true cols=pred", cm.tolist())
    return out

metrics = {
    "sc4001_train_balanced": eval_split(emb_tr, y_tr, "sc4001_train_balanced"),
    "sc4001_val_same_night": eval_split(emb_val, y_val, "sc4001_val_same_night"),
    "sc4002_all_labeled": eval_split(emb2, y2, "sc4002_all_labeled"),
    "sc4002_balanced": eval_split(emb2b, y2b, "sc4002_balanced"),
}
summary = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "meta1_slice_start_sec": meta1.get("slice_start_sec"),
    "meta2_slice_start_sec": meta2.get("slice_start_sec"),
    "history": history,
    "metrics": metrics,
}
(WORKING / "holdout_metrics.json").write_text(json.dumps(summary, indent=2))
torch.save(head.state_dict(), WORKING / "head_a_binary_state_dict.pt")
print("wrote", WORKING / "holdout_metrics.json")


## 6. How to read the numbers

| Split | What it means |
|-------|----------------|
| sc4001_train_balanced | Fit quality on undersampled train night (optimistic) |
| sc4001_val_same_night | Same night, held-out 20% of balanced set |
| sc4002_all_labeled | **True night holdout**, natural class imbalance (mostly drowsy) |
| sc4002_balanced | Holdout night undersampled — fairer comparison to train |

Prefer **macro-F1** + per-class recall on `sc4002_all_labeled`, and **acc/macro-F1** on `sc4002_balanced`, over raw accuracy on the imbalanced all-labeled set.
